In [3]:
#!/usr/bin/env python3
"""
ingest_technical.py

Calcula indicadores técnicos para las ~3.000 empresas del universo
leyendo los precios históricos desde api_raw.multifactor_prices.

CERO llamadas a la API — todo se calcula desde la DB.

Indicadores calculados:
  RSI            → rsi_14_diario, rsi_14_semanal
  Tendencia      → precio_vs_ma200, dist_max_52w
  Volatilidad    → vol_realizada_30d, vol_realizada_90d
  Volumen        → volume_ratio_20d, obv_slope, volume_trend_20d

Output: procesados.multifactor_technical (una fila por ticker por corrida)
"""

import os
import logging
import subprocess
import numpy as np
import pandas as pd
from datetime import datetime, date
from dotenv import load_dotenv
import psycopg2
import psycopg2.extras

# ── ENV ────────────────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD]):
    raise EnvironmentError("Faltan variables de entorno de PostgreSQL")

RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M")
DIAS_MIN = 210    # mínimo de días para calcular MA200 con margen

# ── Logging ────────────────────────────────────────────────────────────────────
LOG_DIR  = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/ingest_technical_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logging.info(f"=== START ingest_technical | run_id: {RUN_ID} ===")

# ── Keep Mac awake ─────────────────────────────────────────────────────────────
try:
    caffeinate = subprocess.Popen(["caffeinate"])
except Exception:
    caffeinate = None

# ── DB ─────────────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD, host=POSTGRES_HOST, port=POSTGRES_PORT
    )

def log_db(conn, ticker, status, message):
    try:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO infraestructura.update_logs
                (schema_name, table_name, ticker, status, message)
                VALUES (%s, %s, %s, %s, %s)
            """, ('procesados', 'multifactor_technical', ticker, status, message))
            conn.commit()
    except Exception:
        conn.rollback()

def obtener_tickers(conn) -> list[str]:
    """Tickers que tienen precios en la DB."""
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT ticker
            FROM api_raw.multifactor_prices
            ORDER BY ticker
        """)
        return [r[0] for r in cur.fetchall()]

def cargar_precios(conn, ticker: str) -> pd.DataFrame:
    """
    Carga el histórico completo de precios para un ticker.
    Devuelve DataFrame con close_adj y volume indexado por fecha.
    """
    sql = """
        SELECT fecha, close_adj, volume
        FROM api_raw.multifactor_prices
        WHERE ticker = %s
          AND close_adj IS NOT NULL
        ORDER BY fecha ASC
    """
    with conn.cursor() as cur:
        cur.execute(sql, (ticker,))
        rows = cur.fetchall()

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows, columns=["fecha", "close_adj", "volume"])
    df["fecha"]     = pd.to_datetime(df["fecha"])
    df["close_adj"] = df["close_adj"].astype(float)
    df["volume"]    = pd.to_numeric(df["volume"], errors="coerce").astype(float)
    df = df.set_index("fecha").sort_index()
    return df


# ── Funciones de cálculo ───────────────────────────────────────────────────────

def calcular_rsi(series: pd.Series, window: int = 14) -> float | None:
    """RSI clásico de Wilder. Devuelve el último valor."""
    s = series.dropna()
    if len(s) < window + 1:
        return None
    delta = s.diff()
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi   = 100 - (100 / (1 + rs))
    val   = rsi.iloc[-1]
    return round(float(val), 2) if not np.isnan(val) else None


def calcular_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """On-Balance Volume acumulado."""
    direccion = np.sign(close.diff())
    return (direccion * volume).fillna(0).cumsum()


def calcular_slope(series: pd.Series, window: int) -> float | None:
    """Pendiente lineal de los últimos `window` puntos. Normalizada por la media."""
    s = series.dropna()
    if len(s) < window:
        return None
    y    = s.iloc[-window:].values.astype(float)
    x    = np.arange(window)
    mean = np.mean(y)
    if mean == 0:
        return None
    slope = np.polyfit(x, y, 1)[0]
    return round(float(slope / mean), 6)   # normalizado → comparable entre tickers


def calcular_vol_anualizada(returns: pd.Series, window: int) -> float | None:
    """Volatilidad realizada anualizada sobre `window` días."""
    r = returns.dropna()
    if len(r) < window:
        return None
    val = float(r.iloc[-window:].std() * np.sqrt(252))
    return round(val, 4) if not np.isnan(val) else None


# ── Cálculo principal por ticker ───────────────────────────────────────────────

def calcular_indicadores(df: pd.DataFrame) -> dict | None:
    """
    Recibe el DataFrame histórico de un ticker y calcula todos los indicadores.
    Devuelve dict con los valores o None si hay datos insuficientes.
    """
    if df.empty or len(df) < DIAS_MIN:
        return None

    close  = df["close_adj"]
    volume = df["volume"]

    # ── Retornos diarios
    returns = close.pct_change()

    # ── RSI diario (14 períodos sobre cierres diarios)
    rsi_diario = calcular_rsi(close, 14)

    # ── RSI semanal (14 períodos sobre cierres semanales)
    close_semanal = close.resample("W-FRI").last().dropna()
    rsi_semanal   = calcular_rsi(close_semanal, 14)

    # ── Tendencia de precio
    precio_hoy = float(close.iloc[-1])

    ma200 = float(close.iloc[-200:].mean()) if len(close) >= 200 else None
    precio_vs_ma200 = round((precio_hoy / ma200 - 1) * 100, 4) if ma200 else None

    max_52w = float(close.iloc[-252:].max()) if len(close) >= 252 else float(close.max())
    dist_max_52w = round((precio_hoy / max_52w - 1) * 100, 4)

    # ── Volatilidad realizada anualizada
    vol_30d = calcular_vol_anualizada(returns, 30)
    vol_90d = calcular_vol_anualizada(returns, 90)

    # ── Volumen — solo si hay datos de volumen
    vol_valido = volume.dropna()
    tiene_vol  = len(vol_valido) >= 20

    if tiene_vol:
        vol_actual     = float(volume.iloc[-1]) if pd.notna(volume.iloc[-1]) else None
        media_vol_20d  = float(volume.iloc[-20:].mean())
        volume_ratio   = round(vol_actual / media_vol_20d, 4) if (vol_actual and media_vol_20d > 0) else None

        # OBV slope — pendiente del OBV normalizada, últimos 20 días
        obv       = calcular_obv(close, volume.fillna(0))
        obv_slope = calcular_slope(obv, 20)

        # Volume trend — pendiente del volumen normalizada, últimos 20 días
        vol_trend = calcular_slope(volume.fillna(0), 20)
    else:
        volume_ratio = None
        obv_slope    = None
        vol_trend    = None

    return {
        "rsi_14_diario":    rsi_diario,
        "rsi_14_semanal":   rsi_semanal,
        "precio_vs_ma200":  precio_vs_ma200,
        "dist_max_52w":     dist_max_52w,
        "vol_realizada_30d":vol_30d,
        "vol_realizada_90d":vol_90d,
        "volume_ratio_20d": volume_ratio,
        "obv_slope":        obv_slope,
        "volume_trend_20d": vol_trend,
    }


# ── INSERT ─────────────────────────────────────────────────────────────────────

INSERT_SQL = """
INSERT INTO procesados.multifactor_technical (
    ticker, fecha_de_consulta,
    rsi_14_diario, rsi_14_semanal,
    precio_vs_ma200, dist_max_52w,
    vol_realizada_30d, vol_realizada_90d,
    volume_ratio_20d, obv_slope, volume_trend_20d,
    run_id
)
VALUES (
    %(ticker)s, %(fecha)s,
    %(rsi_14_diario)s, %(rsi_14_semanal)s,
    %(precio_vs_ma200)s, %(dist_max_52w)s,
    %(vol_realizada_30d)s, %(vol_realizada_90d)s,
    %(volume_ratio_20d)s, %(obv_slope)s, %(volume_trend_20d)s,
    %(run_id)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    rsi_14_diario     = EXCLUDED.rsi_14_diario,
    rsi_14_semanal    = EXCLUDED.rsi_14_semanal,
    precio_vs_ma200   = EXCLUDED.precio_vs_ma200,
    dist_max_52w      = EXCLUDED.dist_max_52w,
    vol_realizada_30d = EXCLUDED.vol_realizada_30d,
    vol_realizada_90d = EXCLUDED.vol_realizada_90d,
    volume_ratio_20d  = EXCLUDED.volume_ratio_20d,
    obv_slope         = EXCLUDED.obv_slope,
    volume_trend_20d  = EXCLUDED.volume_trend_20d,
    updated_at        = CURRENT_TIMESTAMP;
"""


# ── MAIN ───────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*65}")
    print(f"  INGEST TECHNICAL — {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  run: {RUN_ID}")
    print(f"{'='*65}\n")
    print("  Leyendo precios desde api_raw.multifactor_prices...")
    print("  Sin llamadas a la API — cálculo 100% local\n")

    conn    = get_conn()
    tickers = obtener_tickers(conn)
    total   = len(tickers)

    logging.info(f"Tickers a procesar: {total}")

    ok, fail, insuf = 0, 0, 0

    for i, ticker in enumerate(tickers, 1):
        try:
            # Cargar histórico desde DB
            df = cargar_precios(conn, ticker)

            # Calcular indicadores
            indicadores = calcular_indicadores(df)

            if indicadores is None:
                log_db(conn, ticker, "INSUF_DATA",
                       f"Solo {len(df)} días — mínimo {DIAS_MIN}")
                insuf += 1
                continue

            # Insertar
            payload = {
                "ticker": ticker,
                "fecha":  date.today(),
                "run_id": RUN_ID,
                **indicadores,
            }

            with conn.cursor() as cur:
                cur.execute(INSERT_SQL, payload)
            conn.commit()

            log_db(conn, ticker, "success",
                   f"rsi_d={indicadores['rsi_14_diario']} "
                   f"ma200={indicadores['precio_vs_ma200']} "
                   f"vol_ratio={indicadores['volume_ratio_20d']}")
            ok += 1

        except Exception as e:
            conn.rollback()
            log_db(conn, ticker, "exception", str(e))
            fail += 1
            logging.warning(f"  ✗ {ticker}: {e}")

        if i % 200 == 0:
            logging.info(f"  [{i}/{total}] OK={ok} | FAIL={fail} | INSUF={insuf}")

    conn.close()

    print(f"\n{'─'*65}")
    print(f"  OK: {ok} | INSUF. datos: {insuf} | FAIL: {fail}")
    print(f"  Total: {total}")
    print(f"\n{'='*65}")
    print(f"  Pipeline completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")

    logging.info(f"FIN | OK={ok} | INSUF={insuf} | FAIL={fail} | TOTAL={total}")


# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        main()
    finally:
        if caffeinate:
            caffeinate.terminate()

2026-04-08 13:32:09,219 | INFO | === START ingest_technical | run_id: 20260408_1332 ===



  INGEST TECHNICAL — 2026-04-08 13:32  |  run: 20260408_1332

  Leyendo precios desde api_raw.multifactor_prices...
  Sin llamadas a la API — cálculo 100% local



2026-04-08 13:32:09,594 | INFO | Tickers a procesar: 3027
2026-04-08 13:32:10,930 | INFO |   [200/3027] OK=196 | FAIL=0 | INSUF=4
2026-04-08 13:32:12,305 | INFO |   [400/3027] OK=386 | FAIL=0 | INSUF=14
2026-04-08 13:32:13,612 | INFO |   [600/3027] OK=573 | FAIL=0 | INSUF=27
2026-04-08 13:32:14,946 | INFO |   [800/3027] OK=771 | FAIL=0 | INSUF=29
2026-04-08 13:32:16,429 | INFO |   [1000/3027] OK=965 | FAIL=0 | INSUF=35
2026-04-08 13:32:17,943 | INFO |   [1200/3027] OK=1158 | FAIL=0 | INSUF=42
2026-04-08 13:32:19,499 | INFO |   [1400/3027] OK=1355 | FAIL=0 | INSUF=45
2026-04-08 13:32:21,021 | INFO |   [1600/3027] OK=1551 | FAIL=0 | INSUF=49
2026-04-08 13:32:22,652 | INFO |   [1800/3027] OK=1744 | FAIL=0 | INSUF=56
2026-04-08 13:32:24,229 | INFO |   [2000/3027] OK=1939 | FAIL=0 | INSUF=61
2026-04-08 13:32:25,797 | INFO |   [2200/3027] OK=2133 | FAIL=0 | INSUF=67
2026-04-08 13:32:27,298 | INFO |   [2400/3027] OK=2325 | FAIL=0 | INSUF=75
2026-04-08 13:32:28,759 | INFO |   [2600/3027] OK=25


─────────────────────────────────────────────────────────────────
  OK: 2928 | INSUF. datos: 99 | FAIL: 0
  Total: 3027

  Pipeline completo — 13:32:32

